# 🚀 Ultimate Crack Dataset: Dual-Task Starter (Full Train & Val Pipeline)

Welcome to the official starter notebook for the **Ultimate Multi-Domain Surface Defect Dataset**! This notebook provides a complete pipeline showing how to train and validate models for both **Image Classification** (PyTorch ResNet18) and **Object Detection** (Ultralytics YOLOv8).

---

## 🛠️ 1. Setup & Imports

In [ ]:
import os
import cv2
import torch
import pathlib
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from ultralytics import YOLO

## 📁 2. Dataset Paths & Verification

In [ ]:
DATASET_ROOT = pathlib.Path("/kaggle/input/ultimate-crack-dataset")
TRAIN_IMG_DIR = DATASET_ROOT / "images/train"
VAL_IMG_DIR = DATASET_ROOT / "images/val"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 🏷️ 3. Task 1: Image Classification (PyTorch ResNet18 Train & Val)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=str(TRAIN_IMG_DIR), transform=train_transform)
val_dataset = datasets.ImageFolder(root=str(VAL_IMG_DIR), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

num_classes = len(train_dataset.classes)
print(f"Detected {num_classes} classes: {train_dataset.classes}")

In [ ]:
model_clf = models.resnet18(pretrained=True)
model_clf.fc = nn.Linear(model_clf.fc.in_features, num_classes)
model_clf = model_clf.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_clf.parameters(), lr=1e-4)

epochs = 3
for epoch in range(epochs):
    model_clf.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_clf(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        total += labels.size(0)
        correct += preds.eq(labels).sum().item()
    
    train_acc = correct / total
    train_loss = running_loss / total
    
    model_clf.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model_clf(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            val_total += labels.size(0)
            val_correct += preds.eq(labels).sum().item()
            
    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

## 🎯 4. Task 2: Object Detection (YOLO Training & Validation)

In [ ]:
yolo_model = YOLO("yolov8n.pt")

train_results = yolo_model.train(
    data=str(DATASET_ROOT / "data.yaml"),
    epochs=5,
    imgsz=640,
    batch=16,
    project="yolo_crack_benchmark",
    name="train_run"
)

In [ ]:
val_metrics = yolo_model.val()
print(f"mAP50-95: {val_metrics.box.map:.4f}")
print(f"mAP50: {val_metrics.box.map50:.4f}")